# OpenQARP tutorial

This notebook collects code blocks from the [OpenQARP tutorial](https://docs.openqarp.com/source/tutorial.html)

## Hello QARP

In [ ]:
from qarp.blocks import SimpleBlock

bell = SimpleBlock(2, name="bell")
bell.h(0)
bell.cx(0, 1)
bell.measure([(q, q) for q in range(2)])  # measure qubit q into classical bit q
bell.build()

In [ ]:
from qarp.algorithms import Sampler

sampler = Sampler(ket=bell, n_shots=4000)

In [ ]:
from qarp.engines import QarpEngine

engine = QarpEngine(seed=42)
engine.build([sampler])
results = engine.run()

distribution = results[0]
print(distribution)

In [ ]:
print("result:          ", sampler.result)
print("n_qubits:        ", bell.n_qubits)
print("command stream:  ", len(bell.flatten()), "commands")
print("primitive target:", sampler.target)

### Gate builders and the build lifecycle

In [ ]:
import numpy as np
from sympy import Symbol

theta = Symbol("theta")

demo = SimpleBlock(3, name="demo")
demo.h(0)
demo.cx(0, 1)
demo.ry(2, theta)  # parametric gates accept sympy Symbols (or floats)
demo.rz(2, 0.25)  # angles are ALWAYS in radians
demo.build()

In [ ]:
print("symbols:", demo.symbols)
for cmd in demo.flatten():
    print(cmd)

### Composition

In [ ]:
from qarp.blocks import (
    CompositeBlock,
    ComputationalBasisStateBlock,
    HEABlock,
    ReadoutBlock,
)

prep = ComputationalBasisStateBlock([1, 1, 0, 0])  # X on qubits 0, 1
hea = HEABlock(4, 2, True, True, True, False)  # n_qubits, n_layers, real, linear, circular, use_cz
readout = ReadoutBlock(n_qubits=4)  # measure all qubits, cbit q <- qubit q

circuit = CompositeBlock([prep, hea, readout]).build()

### Circuit visualization

In [ ]:
circuit.plot()
#circuit.plot(decompose_boxes=True)  # decompose boxes to see the underlying gates

### Setting parameter values

In [ ]:
bound = circuit.set_symbols(circuit.parameter_map([0.1] * len(circuit.symbols)))

print("original:", circuit.flatten()[2:5])  # Ry gates, still symbolic
print("bound:   ", bound.flatten()[2:5])  # same gates, values applied
print("structure is unchanged:", bound.build().symbols)

In [ ]:
flip = SimpleBlock(1)
flip.ry(0, np.pi)
flip.measure(0, 0)
flip.build()

flip_sampler = Sampler(ket=flip, n_shots=1000)
flip_engine = QarpEngine(seed=1)
flip_engine.build([flip_sampler])
print(flip_engine.run()[0])

### Bit order

In [ ]:
from qarp.endianness import bits_to_label, label_to_bits

bits = (0, 1, 1)  # qubit 0 = 0, qubit 1 = 1, qubit 2 = 1
print(bits_to_label(bits))  # 2**1 + 2**2 = 6
print(label_to_bits(6, 3))

### The target is inferred from what you pass

In [ ]:
from qarp.algorithms import StateVector
from qarp.blocks import HnBlock
from qarp.operators import QubitOperator

psi = HEABlock(3, 2, True, True, True, False).build()
psi_01 = psi.set_symbols(psi.parameter_map([0.1] * len(psi.symbols))).build()
plus = HnBlock(n_qubits=3).build()

H = QubitOperator("Z0 Z1", -1.0) + QubitOperator("X2", 0.5)

for prim in (
    Sampler(ket=psi_01),
    StateVector(bra=plus, ket=psi_01),
    StateVector(bra=psi_01, operator=H, ket=psi_01),
    StateVector(bra=plus, operator=H, ket=psi_01),
):
    prim.build()
    print(f"{type(prim).__name__:12s} -> {prim.target}")

### Exact expectation values

In [ ]:
expval = StateVector(bra=psi, operator=H, ket=psi)  # symbolic: values at run time

ev_engine = QarpEngine()
ev_engine.build([expval])

param_values = psi.parameter_map(np.linspace(0.0, 1.0, len(psi.symbols)))
print("<psi|H|psi> =", ev_engine.run(param_values)[0])
print("stored on the primitive too:", expval.result)

### Exact but dense matrix vector approach

In [ ]:
Hmat = expval.operator.sparse_matrix()
psivec = psi.set_symbols(param_values).build().statevector()

print(psivec.T.conj() @ Hmat @ psivec)  # <psi|H|psi> = psi^dagger H psi

### Overlaps

In [ ]:
overlap = StateVector(bra=plus, ket=psi_01)
ov_engine = QarpEngine()
ov_engine.build([overlap])
print("<+++|psi(0.1)> =", ov_engine.run()[0])

### Sampling and shot-based estimation

In [ ]:
from qarp.algorithms import TermwiseHadamardTest

exact_ev = StateVector(bra=psi_01, operator=H, ket=psi_01)
estimated_ev = TermwiseHadamardTest(bra=psi_01, operator=H, ket=psi_01, n_shots=20_000)

shot_engine = QarpEngine(seed=7)
shot_engine.build([exact_ev, estimated_ev])  # one engine, several primitives
res = shot_engine.run()
print("exact:    ", res[0].real)
print("estimated:", res[1].real)

## Engines: compile once, run many

In [ ]:
eng_ansatz = HEABlock(4, 2, True, True, True, False).build()
eng_H = QubitOperator("Z0 Z1") + QubitOperator("Z2 Z3") + QubitOperator("X0", 0.3)

eng_expval = StateVector(bra=eng_ansatz, operator=eng_H, ket=eng_ansatz)

sweep_engine = QarpEngine()
sweep_engine.build([eng_expval])  # compile once...

for angle in (0.0, 0.3, 0.6):  # ...run many
    params = eng_ansatz.parameter_map([angle] * len(eng_ansatz.symbols))
    print(f"theta={angle:.1f}  <H> = {sweep_engine.run(params)[0].real:+.6f}")

### Reproducibility

In [ ]:
for attempt in range(2):
    repeat_sampler = Sampler(ket=bell, n_shots=100)
    repeat_engine = QarpEngine(seed=123)
    repeat_engine.build([repeat_sampler])
    print(repeat_engine.run()[0])

### Parameter sweeps with ``batch_run``

In [ ]:
param_sets = [
    eng_ansatz.parameter_map([t] * len(eng_ansatz.symbols)) for t in np.linspace(0, 2 * np.pi, 9)
]

batch_engine = QarpEngine()
batch = batch_engine.batch_run([eng_expval], param_sets)
for ps, row in zip(param_sets, batch, strict=True):
    print(f"theta={list(ps.values())[0]:+.3f}  <H> = {row[0]:+.6f}")

### Analytic gradients

In [ ]:
grad_engine = QarpEngine()
grad_engine.build([eng_expval])

grad_params = eng_ansatz.parameter_map([0.4] * len(eng_ansatz.symbols))
grad = grad_engine.run_gradient(grad_params)[0]
print("dE/dtheta_k:", np.round(grad, 6))

## Your own variational loop

In [ ]:
n_spins = 3
field = 0.5
H_tfim = QubitOperator()
for i in range(n_spins - 1):
    H_tfim += QubitOperator(f"Z{i} Z{i + 1}", -1.0)
for i in range(n_spins):
    H_tfim += QubitOperator(f"X{i}", -field)

# Exact reference from the dense matrix (LSB, like everything else).
exact_energy = np.linalg.eigvalsh(H_tfim.sparse_matrix().toarray())[0]
print("exact ground energy:", exact_energy)

### The four ingredients

In [ ]:
from qarp.optimizers import ScipyOptimizer

tfim_ansatz = HEABlock(n_spins, 2, True, True, True, False).build()
tfim_expval = StateVector(bra=tfim_ansatz, operator=H_tfim, ket=tfim_ansatz)

tfim_engine = QarpEngine()
tfim_engine.build([tfim_expval])  # compile once, before the loop

print(len(tfim_ansatz.symbols), "parameters")

### The objective function

In [ ]:
evals = []

def energy(x):
    value = tfim_engine.run(tfim_ansatz.parameter_map(x))[0].real
    evals.append(value)
    return value

### Minimise

In [ ]:
rng = np.random.default_rng(42)
x0 = rng.uniform(0, 2 * np.pi, len(tfim_ansatz.symbols))

opt = ScipyOptimizer("COBYLA", options={"maxiter": 400})
result = opt.minimize(energy, x0)

print(f"VQE energy:   {result.fun:.6f}")
print(f"exact energy: {exact_energy:.6f}")
print(f"error:        {result.fun - exact_energy:.2e}   ({len(evals)} evaluations)")

### With gradients

In [ ]:
def gradient(x):
    return tfim_engine.run_gradient(tfim_ansatz.parameter_map(x))[0]


result_cg = ScipyOptimizer("CG").minimize(energy, x0, gradient=gradient)
print(f"CG energy:    {result_cg.fun:.6f}   error {result_cg.fun - exact_energy:.2e}")

### The reveal: this *is* ``VQE``

In [ ]:
from qarp.algorithms import VQE

vqe = VQE(
    operator=H_tfim,
    ket=HEABlock(n_spins, 2, True, True, True, False).build(),
    initial_parameters=x0,
    optimizer=ScipyOptimizer("COBYLA", options={"maxiter": 400}),
    primitive=StateVector(),
    engine=QarpEngine(),
    verbose=False,
)
vqe.build()
vqe_energy, vqe_params = vqe.run()
print(f"VQE composite: {vqe_energy:.6f}   error {vqe_energy - exact_energy:.2e}")

### One complete run: QAOA on MaxCut

In [ ]:
import networkx as nx

from qarp.algorithms import QAOA

G = nx.gnm_random_graph(6, 9, seed=7)
for u, v in G.edges:
    G[u][v]["weight"] = 1

qaoa = QAOA(
    problem=G,
    n_layers=2,
    use_rzz=False,
    initial_parameters=[0.1] * 4,  # 2 angles per layer
    optimizer=ScipyOptimizer("COBYLA", options={"maxiter": 150}),
    verbose=False,
).build()

qaoa.run()

# The optimised ansatz, the optimum and the optimisation record are all on the object.
print("best energy: ", qaoa.result.fun)
print("n parameters:", len(qaoa.result.x))

## Where to find more tutorials

This notebook is only an introduction. More runnable tutorials are available in the [`examples/` directory of the official public OpenQARP repository](https://github.com/OpenQARP/openqarp/tree/develop/examples). They demonstrate the different building blocks, algorithms, execution mechanisms, and analysis tools provided by QARP.

For more realistic scientific applications, explore [`examples/use_cases/`](https://github.com/OpenQARP/openqarp/tree/develop/examples/use_cases), which contains complete use cases showing how these mechanisms can be combined in practical workflows.